# Continuous Integration — Automated Tests on Every Change

## Objective
The Level 2 (CI/CD pipeline automation) notebook. Level 1 (NB1–NB4) is the hand-run loop; Level 2 tests changes and delivers automatically. For example, a pull request runs tests (CI, this notebook) and the deployment pipeline (CD NB6). The coupled CI/CD sequence and PR trigger is set-up for demonstration purposes. Users can customize to their organization standards.

This notebook writes the test suite into `tests/` (via `%%writefile`) and runs it. The same files are committed in the repo, so the GitHub Actions / Terraform route works without running this notebook.

Note: CI needs no Google Cloud credentials and no WIF — these are unit/mock tests (config, schemas, pure functions). The keyless WIF auth + the `cicd.github_repo` config are only for CD (deploying) — set up in NB6.

## CI Tests

Every test, its category, and where it lives. Central framework code is shared for all agents. Agent-specific is one particular agent's own rules.

| Category | Test | Repo path | What it checks (plain English) |
|---|---|---|---|
| Central | `test_parse_verdict` | `tests/test_framework.py` | We can pull the PASS/FAIL verdict out of the model's reply, even when it's wrapped in code fences or extra text |
| Central | `test_thresholds_*` | `tests/test_framework.py` | The pass/fail score thresholds are read correctly from config |
| Central | `test_rubric_lines_*` | `tests/test_framework.py` | The grading rubric is turned into the right text for the judge prompt |
| Central | `test_at_least_one_agent_discovered` | `tests/test_agents.py` | At least one agent was found to test |
| Central | `test_agent_yaml_is_valid[<agent>]` | `tests/test_agents.py` | Each agent's config file has all the required fields, with sensible values |
| Central | `test_golden_evalset_schema[<agent>]` | `tests/test_agents.py` | Each agent's eval dataset is well-formed — unique cases, valid verdicts, and both pass & fail examples |
| Central | `test_defaults_yaml_has_model_and_project` | `tests/test_agents.py` | The repo's default config sets a model and a project |
| Central | `test_agent_runs_under_dedicated_per_agent_sa` | `tests/test_identity.py` | Every agent runs under its own service account, not a shared/default one |
| Central | `test_agent_sa_roles_within_allowlist` | `tests/test_identity.py` | The agent's permissions stay within a reviewed least-privilege allowlist |
| Central | `test_deployer_sa_roles_within_allowlist` | `tests/test_identity.py` | The CI/CD deployer's permissions stay within its reviewed allowlist |
| Central | `test_no_project_owner_or_editor` | `tests/test_identity.py` | No identity is granted project Owner/Editor |
| Central | `test_no_long_lived_service_account_keys` | `tests/test_identity.py` | No long-lived service-account keys (keyless: WIF + Agent Identity) |
| Agent-specific | `test_instruction_enumerates_all_five_rules` | `tests/agents/test_brand_guidelines.py` | The agent's prompt actually lists all 5 brand rules |
| Agent-specific | `test_instruction_declares_json_verdict_contract` | `tests/agents/test_brand_guidelines.py` | The prompt asks for a PASS/FAIL answer in JSON |
| Agent-specific | `test_golden_exercises_every_brand_rule` | `tests/agents/test_brand_guidelines.py` | The eval set has an example that breaks each of the 5 rules |
| Agent-specific | `test_weak_baseline_keeps_contract_but_drops_rules` | `tests/agents/test_brand_guidelines.py` | The deliberately "drifted" prompt still returns valid JSON but no longer lists the rules |

The `tests/test_identity.py` rows are the **Agent Identity & least-privilege** lint. They check the IAM *config* (dedicated per-agent SA, role allowlists, no long-lived keys); the runtime SPIFFE identity itself is auto-issued at deploy and verified there.

`tests/conftest.py` (no tests) puts the repo root + `eval_tool/` on `sys.path`. CI needs no cloud creds / no WIF.

### `tests/conftest.py` — `sys.path` setup (no tests)

In [1]:
%%writefile ../tests/conftest.py
"""Test setup — puts the repo root + eval_tool/ on sys.path so the suite can import the
loop/eval modules.

Two categories:
  * Central tests (test_framework.py + test_agents.py) — the eval/loop CODE every agent uses
    AND the generic contract validated against EVERY agents/*/ (add an agent and it's checked
    automatically). Nothing here is tied to one agent.
  * Agent-specific tests (tests/agents/<agent>.py) — one particular agent's own rules; add a
    file here only when an agent needs more than the central contract.

All UNIT / mock — no Google Cloud calls — so CI runs them on every PR without credentials.
"""
import os
import sys

ROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
for _p in (ROOT, os.path.join(ROOT, "eval_tool")):
    if _p not in sys.path:
        sys.path.insert(0, _p)


Overwriting ../tests/conftest.py


### `tests/test_framework.py` — Central tests (framework code)

In [2]:
%%writefile ../tests/test_framework.py
"""Central tests — framework code.

Exercise the eval/loop code that EVERY agent uses (eval_tool/), independent of which agents
exist. Imports the modules (no cloud calls).
"""
import pytest

import managed_eval
import run_eval


@pytest.mark.parametrize("text,expected", [
    ('{"verdict": "PASS"}', "PASS"),
    ('{"verdict": "FAIL", "violations": [{"rule": 1}]}', "FAIL"),
    ('```json\n{"verdict": "FAIL"}\n```', "FAIL"),       # fenced -> regex fallback
    ('Sure! {"verdict":"PASS"} done', "PASS"),            # prose-wrapped -> regex fallback
    ('no verdict here', None),
    ('', None),
    (None, None),                                         # must not raise (TypeError-safe)
])
def test_parse_verdict(text, expected):
    assert run_eval.parse_verdict(text) == expected


def test_thresholds_defaults_when_missing():
    assert managed_eval.thresholds({}) == (0.8, 1.0)


def test_thresholds_reads_explicit_values():
    crit = {"criteria": {
        "rubric_based_final_response_quality_v1": {"threshold": 0.7},
        "safety_v1": {"threshold": 0.9},
    }}
    assert managed_eval.thresholds(crit) == (0.7, 0.9)


def test_rubric_lines_fallback_is_nonempty():
    assert managed_eval._rubric_lines({}).strip()


def test_rubric_lines_renders_bullets():
    crit = {"criteria": {"rubric_based_final_response_quality_v1": {"rubrics": [
        {"rubric_id": "cites_rule", "rubric_content": {"textProperty": "cites the broken rule"}},
    ]}}}
    out = managed_eval._rubric_lines(crit)
    assert "cites_rule" in out and "cites the broken rule" in out


Overwriting ../tests/test_framework.py


### `tests/test_agents.py` — Central tests (generic contract, every `agents/*/`)

In [3]:
%%writefile ../tests/test_agents.py
"""Central tests — the generic agent contract.

Discovered + parametrized over EVERY agents/*/agent.yaml, so dropping in a new agent folder
makes it validated automatically (each agent shows as its own test case). Not tied to any one
agent. Plus the repo-level configs/defaults.yaml. Pure (PyYAML / JSON); no cloud calls. An agent
that needs more than this central contract adds its own file under tests/agents/.
"""
import glob
import json
import os

import pytest
import yaml

ROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
AGENT_DIRS = sorted(os.path.dirname(p) for p in glob.glob(os.path.join(ROOT, "agents", "*", "agent.yaml")))
AGENT_IDS = [os.path.basename(d) for d in AGENT_DIRS]
GOLDEN = [(os.path.basename(d), os.path.join(d, "eval", "golden.evalset.json"))
          for d in AGENT_DIRS if os.path.exists(os.path.join(d, "eval", "golden.evalset.json"))]


def _yaml(path):
    with open(path) as f:
        return yaml.safe_load(f)


def test_at_least_one_agent_discovered():
    assert AGENT_DIRS, "no agents/*/agent.yaml found"


@pytest.mark.parametrize("agent_dir", AGENT_DIRS, ids=AGENT_IDS)
def test_agent_yaml_is_valid(agent_dir):
    cfg = _yaml(os.path.join(agent_dir, "agent.yaml"))
    name = os.path.basename(agent_dir)
    for key in ("name", "display_name", "model", "eval", "monitor", "deploy"):
        assert key in cfg, f"{name}/agent.yaml missing required key: {key}"
    assert isinstance(cfg["model"], str) and cfg["model"].strip(), f"{name}: model unset"
    assert int(cfg["eval"].get("num_runs", 0)) >= 1, f"{name}: eval.num_runs must be >= 1"
    assert cfg["eval"].get("engine") in ("managed", "local"), f"{name}: eval.engine invalid"
    assert 0.0 < float(cfg["monitor"]["drift_threshold"]) <= 1.0, f"{name}: drift_threshold out of range"


@pytest.mark.parametrize("name,path", GOLDEN, ids=[n for n, _ in GOLDEN])
def test_golden_evalset_schema(name, path):
    cases = json.load(open(path))["eval_cases"]
    assert cases, f"{name}: empty golden set"
    ids = [c["id"] for c in cases]
    assert len(ids) == len(set(ids)), f"{name}: duplicate case ids"
    for c in cases:
        assert "id" in c and "input" in c and "expected" in c, f"{name}/{c.get('id')}: missing fields"
        assert c["expected"].get("verdict") in ("PASS", "FAIL"), f"{name}/{c['id']}: bad verdict"
        if c["expected"]["verdict"] == "FAIL":
            viols = c["expected"].get("violations", [])
            assert viols and all("rule" in v for v in viols), f"{name}/{c['id']}: FAIL must cite rules"
    assert {c["expected"]["verdict"] for c in cases} == {"PASS", "FAIL"}, f"{name}: cover both verdicts"


def test_defaults_yaml_has_model_and_project():
    d = _yaml(os.path.join(ROOT, "configs", "defaults.yaml"))
    assert d["defaults"]["model"]
    assert d["project"]["id"]


Overwriting ../tests/test_agents.py


### `tests/agents/test_brand_guidelines.py` — Agent-specific tests

In [4]:
%%writefile ../tests/agents/test_brand_guidelines.py
"""Agent-specific tests — the brand-guidelines-checker's own domain rules, BEYOND the central
contract in tests/test_agents.py. The pattern for a single agent: add a file here only when an
agent needs more than the central checks.

Pure: imports agent.py and reads the agent's files; no Google Cloud calls.
"""
import importlib.util
import json
import os

ROOT = os.path.dirname(os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
AGENT_DIR = os.path.join(ROOT, "agents", "brand-guidelines-checker")

# This agent encodes exactly five brand rules — that count is what makes these tests agent-specific.
BRAND_RULES = (1, 2, 3, 4, 5)


def _agent_module():
    spec = importlib.util.spec_from_file_location("brand_agent", os.path.join(AGENT_DIR, "app", "agent.py"))
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod


def _golden_cases():
    with open(os.path.join(AGENT_DIR, "eval", "golden.evalset.json")) as f:
        return json.load(f)["eval_cases"]


def test_instruction_enumerates_all_five_rules():
    instr = _agent_module().DEFAULT_INSTRUCTION
    for n in BRAND_RULES:
        assert f"{n}." in instr, f"brand rule {n} is not enumerated in the instruction"


def test_instruction_declares_json_verdict_contract():
    instr = _agent_module().DEFAULT_INSTRUCTION
    assert "verdict" in instr and "PASS" in instr and "FAIL" in instr


def test_golden_exercises_every_brand_rule():
    cited = {v["rule"] for c in _golden_cases() for v in c["expected"].get("violations", [])}
    missing = set(BRAND_RULES) - cited
    assert not missing, f"golden set never exercises brand rule(s): {sorted(missing)}"


def test_weak_baseline_keeps_contract_but_drops_rules():
    # The deliberately-drifted baseline must stay parseable (keeps the verdict contract) yet NOT
    # spell out the rules — that's the premise that lets GEPA rediscover them in NB3.
    weak = open(os.path.join(AGENT_DIR, "eval", "weak_instruction.txt")).read()
    assert "verdict" in weak.lower(), "weak instruction must keep the JSON verdict contract"
    assert not all(f"{n}." in weak for n in BRAND_RULES), "weak instruction should NOT list all rules"


Overwriting ../tests/agents/test_brand_guidelines.py


### `tests/test_identity.py` — Central tests (Agent Identity & least-privilege)

In [5]:
%%writefile ../tests/test_identity.py
"""Agent Identity & least-privilege — Central CI lint.

The agent's RUNTIME identity (Agent Identity / SPIFFE) is auto-issued by Agent
Runtime at deploy time and can't be unit-tested. What we CAN enforce in CI,
credential-free, is the least-privilege CONFIG around it: every agent runs under
its own dedicated service account, granted roles stay within a reviewed allowlist
(no owner/editor/broad-admin), and there are no long-lived service-account keys
(keyless = WIF + Agent Identity).

Deny-by-default: adding a role to the Terraform means adding it to the allowlist
here too — a deliberate review gate.
"""
import re
from pathlib import Path

REPO = Path(__file__).resolve().parents[1]
AGENT_IAM = REPO / "terraform/modules/agent_ops/iam.tf"
CICD_WIF = REPO / "cicd/wif.tf"

# Reviewed least-privilege allowlist for the per-agent ops SA.
AGENT_SA_ROLES = {
    "roles/aiplatform.user",
    "roles/logging.logWriter",
    "roles/cloudtrace.agent",
    "roles/monitoring.metricWriter",
    "roles/storage.objectAdmin",
}

# Reviewed allowlist for the CI/CD deployer SA (broader by necessity — it builds + deploys).
DEPLOYER_SA_ROLES = {
    "roles/aiplatform.user",
    "roles/artifactregistry.writer",
    "roles/storage.admin",
    "roles/cloudbuild.builds.editor",
    "roles/logging.viewer",
    "roles/iam.workloadIdentityUser",
}

# Never acceptable on a non-human identity.
FORBIDDEN = {"roles/owner", "roles/editor"}


def _roles(path):
    return set(re.findall(r"roles/[A-Za-z0-9._]+", path.read_text()))


def test_agent_runs_under_dedicated_per_agent_sa():
    # Each agent gets its own {name}-sa (folder-derived), never a shared/default SA.
    assert '"${var.agent_name}-sa"' in AGENT_IAM.read_text()


def test_agent_sa_roles_within_allowlist():
    extra = _roles(AGENT_IAM) - AGENT_SA_ROLES
    assert not extra, f"agent SA has roles outside the least-privilege allowlist: {sorted(extra)}"


def test_deployer_sa_roles_within_allowlist():
    extra = _roles(CICD_WIF) - DEPLOYER_SA_ROLES
    assert not extra, f"deployer SA has roles outside its reviewed allowlist: {sorted(extra)}"


def test_no_project_owner_or_editor():
    for f in (AGENT_IAM, CICD_WIF):
        assert not (_roles(f) & FORBIDDEN), f"{f.name} grants project owner/editor (not least-privilege)"


def test_no_long_lived_service_account_keys():
    for f in (AGENT_IAM, CICD_WIF):
        assert "google_service_account_key" not in f.read_text(), (
            f"{f.name} creates a long-lived SA key — use WIF / Agent Identity instead"
        )


Overwriting ../tests/test_identity.py


## Run the Suite (parity with CI)

Same command the GitHub Action runs — from the repo root so `conftest` resolves imports. No cloud calls, so no credentials needed. Each agent appears as its own parametrized case.

In [6]:
!cd .. && python -m pytest tests -v

============================= test session starts ==============================
platform linux -- Python 3.10.13, pytest-9.0.3, pluggy-1.6.0 -- /opt/conda/bin/python
cachedir: .pytest_cache
rootdir: /home/jupyter/tmp/support/20240301_181108/jupyter/@ME/agent-ops-maturity/agent-ops-maturity-repotest
plugins: typeguard-4.1.5, anyio-4.9.0
collected 24 items                                                             

tests/agents/test_brand_guidelines.py::test_instruction_enumerates_all_five_rules PASSED [  4%]
tests/agents/test_brand_guidelines.py::test_instruction_declares_json_verdict_contract PASSED [  8%]
tests/agents/test_brand_guidelines.py::test_golden_exercises_every_brand_rule PASSED [ 12%]
tests/agents/test_brand_guidelines.py::test_weak_baseline_keeps_contract_but_drops_rules PASSED [ 16%]
tests/test_agents.py::test_at_least_one_agent_discovered PASSED          [ 20%]
tests/test_agents.py::test_agent_yaml_is_valid[brand-guidelines-checker] PASSED [ 25%]
tests/test_agents.py:

## How These Run in CI

These tests are the `ci.yml` reusable workflow (`.github/workflows/ci.yml`) — a standalone file you edit independently. The thin `pipeline.yml` orchestrator (set up in NB6) calls it on a PR to `main`, then runs the reusable `cd.yml` only if it passes (`cd needs ci`). So CI and CD are separate files, run in sequence — production-style, not one monolithic workflow. The CI workflow:

In [7]:
!cat ../.github/workflows/ci.yml

# CI — reusable workflow (called by pipeline.yml; not triggered on its own).
# Unit/mock tests only: config + schema validation + pure functions. No Google Cloud creds.
# Edit this file to change what "CI" means; the orchestrator (pipeline.yml) decides WHEN it runs.
name: CI

on:
  workflow_call:

jobs:
  unit-tests:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: "3.11"
          cache: pip
      - run: python -m pip install --upgrade pip
      - run: pip install -r requirements-dev.txt
      - name: Lint
        run: ruff check tests/
      - name: Unit tests
        run: pytest tests/ -v


## Recap & Next

- CI = tests on every PR to `main`, in two categories: Central (framework + the generic contract over every `agents/*/`) and Agent-specific (one agent's own rules).
- They live in the reusable `.github/workflows/ci.yml`, which the `pipeline.yml` orchestrator runs before `cd.yml` (`cd needs ci`) on a PR — separate files, sequenced. Nothing runs on push.

Next: `06-cd-pipeline.ipynb` — provision WIF (the `cicd/` Terraform) and set the repo variables the committed `pipeline.yml` reads, which runs `ci.yml` → `cd.yml` on a PR.